In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Create pareto front from all multiple runs 

In [ ]:
def read_sm(sm_path, soil_depth,i, dates):
    
    df = pd.read_csv(sm_path, skiprows=20, header=[0, 1], sep='\t')
    df.columns = [col[0] if 'Unnamed' in col[1] else ' '.join(col) for col in df.columns.values]
    df['Date'] = pd.to_datetime(df.rename(columns={'mday': 'day'})[['year', 'month', 'day']])
    df = df[df['Date'].isin(dates)]
    # print(df)
    df = df.loc[:, ['Date', 'Matrix water mm']]
    df[f'sim_sm_perc_{i}'] = df['Matrix water mm'] / soil_depth
    # df = df.loc[(df['Date'] > pd.to_datetime(start_date) 
    #              & (df['Date'] < pd.to_datetime(end_date))]
    
    # df = df.loc[:,['Date', f'sim_sm_perc_{i}']]
    #print(df)
    sm = df[f'sim_sm_perc_{i}'].to_numpy()
    return sm

def read_harvest(harvest_dlf):
    df_harvest = pd.read_csv(harvest_dlf, skiprows = 9, header = [0,1], sep = '\t')
    df_harvest.columns = [col[0] if 'Unnamed' in col[1] else ' '.join(col) for col in df_harvest.columns.values]
    df_harvest = df_harvest[df_harvest['year'] > 2005]
    harvest = df_harvest['sorg_DM Mg DM/ha'].to_numpy()
    return harvest

def read_irrigation(irrigation_dlf):
    df_irrigate = pd.read_csv(irrigation_dlf, skiprows = 26, header = [0,1], sep = '\t')
    df_irrigate.columns = [col[0] if 'Unnamed' in col[1] else ' '.join(col) for col in df_irrigate.columns.values]
    #print(df_irrigate)
    df_irrigate = df_irrigate[['year', 'month' , 'mday','Irrigation mm']]
    #df_irrigate = df_irrigate.iloc[5:].reset_index(drop= True)
    
    #winter_wheat_specific_dates
    # df_irrigate['crop_year'] = np.where(df_irrigate['month'] <= 8, df_irrigate['year'],df_irrigate['year']+1)
    # df_irrigate = df_irrigate[df_irrigate['crop_year'] > 2005]
    # df_irrigate = df_irrigate[df_irrigate['crop_year'] != 2012]
    
    #sugar_beet_specific_dates
    #print(df_irrigate)
    years = [y for y in range(2008,2023) if y!= 2012]
    df_irrigate['crop_year'] = np.where(df_irrigate['month'] <= 12, df_irrigate['year'],df_irrigate['year'])
    df_irrigate = df_irrigate[df_irrigate['crop_year'].isin(years)]
    # df_irrigate = df_irrigate[df_irrigate['crop_year'] > 2009]
    # df_irrigate = df_irrigate[df_irrigate['crop_year'] != 2017]
    
    df_irrigate = df_irrigate.groupby('crop_year')['Irrigation mm'].sum(min_count=1).reset_index()
    #print(df_irrigate)
    irrigation = df_irrigate['Irrigation mm'].to_numpy()
    return irrigation

def write_csv(np_array, col_names, save_path):
    df = pd.DataFrame(np_array, columns = col_names)
    df.to_csv(save_path, index = True)
    #print(df.head())
def read_n_save_outputs_np(main_dir, save = True):
    years = [y for y in range(2008,2023) if y!= 2012]
    yield_outputs = []
    irrigation_outputs = []
    sm_outputs = []
    parameters = []

    df_obs = pd.read_excel(r'C:\Users\Adhikari\Desktop\wwheat_stepwise\winterwheat_sm30.xlsx')
    df_obs30 = df_obs.loc[:,['Date', 'obs moist_30 cm', 'year']]
    df_obs30 = df_obs[df_obs30['year'].isin(years)]
    # print(df_obs30)
    sm_dates = df_obs30['Date'].to_numpy()
    print(sm_dates, sm_dates.shape)
    # print(sm_dates)
    # print(sm_dates.type())
    for i in range(0,len(os.listdir(os.path.join(main_dir, 'save_dir')))):
        
        #reates file path of output to read
        parameter_txt = os.path.join(main_dir, 'save_dir', str(i),'parameters.txt')
        harvest_dlf = os.path.join(main_dir, 'save_dir', str(i),'harvest.dlf')
        irrigation_dlf = os.path.join(main_dir, 'save_dir', str(i),'field_water.dlf')
        sm30_dlf = os.path.join(main_dir, 'save_dir', str(i),'soil_water30cm.dlf')
        
        #parameter
        df_parameter = pd.read_csv(parameter_txt, header = [0], sep = ';')
        if i == 0:
            parameter_names = df_parameter.columns.to_list()
            print(len(parameter_names))
            print(parameter_names)
        # print(df_parameter.to_numpy().shape)
        # print(len(df_parameter.to_numpy()[0,:]))
        # if df_parameter.to_numpy().shape[1] != 89:
        #     print(i)
        
        parameters.append(df_parameter.to_numpy()[0,:])
        
        #harvest
        yield_this_model_run = read_harvest(harvest_dlf)
        yield_outputs.append(yield_this_model_run)
        
        #irrigate
        irrigation_this_model_run = read_irrigation(irrigation_dlf)
        irrigation_outputs.append(irrigation_this_model_run)
        
        #soil_moisture
        sm_30_this_model_run = read_sm(sm30_dlf, 300,i, sm_dates)
        sm_outputs.append(sm_30_this_model_run)
        #     #print(df_irrigate)
        # #remove the first 5 years of output if required
        print(i, end = '; ')
        # if len(yield_this_model_run) < 14:
        #         print('Model structure is not stable.')
        # if i == 1:
        #     break
    
    #compile all read outputs
    parameter_array = np.array(parameters)
    yield_array = np.array(yield_outputs)
    irrigation_array = np.array(irrigation_outputs)
    sm_array = np.array(sm_outputs)
   
    #wtite them into a csv
    if save == True:
        write_csv(parameter_array, parameter_names, os.path.join(main_dir, 'parameters_combined.txt'))
        write_csv(yield_array, years, os.path.join(main_dir, 'yield.txt'))
        write_csv(irrigation_array, years, os.path.join(main_dir, 'irrigation.txt'))
        write_csv(sm_array, sm_dates, os.path.join(main_dir, 'sm.txt'))
    return parameter_array, yield_array, irrigation_array, sm_array
a,b,c,d = read_n_save_outputs_np(r'C:\Users\Adhikari\Desktop\wwheat_stepwise\2_a_wwheat_nsga\pareto final', True)

Plot the Pareto front

In [ ]:
def plot_pareto(file_path = r'C:\Users\Adhikari\Desktop\sbeet_stepwise\2_nsgaII\final pareto\yield.txt', 
                sheet_name = 'sugarbeet_optimal',
                y_label = 'Crop Yield [Mg DM/ha]', # 'Irrigation amount [mm]'
                save_path = r'C:\Users\Adhikari\Desktop\sbeet_stepwise\2_nsgaII\final pareto\ensemble_yld.png'):
    yld = pd.read_csv(file_path, index_col=0)
    yld_np = yld.to_numpy()
    yld_mean = np.mean(yld_np, axis = 0)
    yld_median = np.quantile(yld_np, axis = 0, q = 0.5)
    # plt.plot(yld.columns, yld_mean)
    # plt.plot(yld.columns, yld_median)
    
    #read all reference values, pareto front simulations.
    df_obsyld = pd.read_excel(r'C:\Users\Adhikari\Desktop\Thesis\Thesis Shared Cloud\all_crop_field_results.xlsx', header = [0], sheet_name = sheet_name)
    obsyld = df_obsyld['Crop_yield[Mg DM/ha]']
    yld_min = np.min(yld_np, axis=0)
    yld_max = np.max(yld_np, axis=0)
    yld_mean = np.mean(yld_np, axis=0)
    yld_median = np.quantile(yld_np, axis=0, q=0.5)

    plt.figure(figsize=(16, 7))

    # Plot min-max shaded region
    plt.fill_between(yld.columns, yld_min, yld_max, 
                    color='lightblue', alpha=0.3, label='Min-Max Range')

    # Plot mean and median lines
    plt.plot(yld.columns, yld_mean, color='#2E86AB', linewidth=3, 
            marker='o', markersize=4, label='Mean')
    plt.plot(yld.columns, yld_median, color='#A23B72', linewidth=3, 
            marker='s', markersize=4, label='Median', linestyle='--')

    # Plot observed data
    plt.plot(yld.columns, obsyld, color='#F18F01', linewidth=3, 
            marker='^', markersize=6, label='Observed', linestyle='-')

    plt.xlabel('Years', fontsize=12, fontweight='bold')
    plt.ylabel(y_label, fontsize=12, fontweight='bold')
    # plt.title('Sugar Beet Yield: Simulated vs Observed', fontsize=14, fontweight='bold')
    plt.ylim(0, max(yld_max))
    
    plt.yticks(range(0,26,2)) #this can be adjusted based on the range of the y-axis values


    plt.xticks(rotation=45, ha='right')
    plt.axvline(x=7.5, color='red', linestyle='--', linewidth=2, 
                alpha=0.7, label='Validation Period Start')
    
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(frameon=True, fancybox=True, shadow=True, loc='lower right')
    plt.savefig(save_path)
    

    plt.tight_layout()
    plt.show()